## Loading Environment Variables from the secrets file

In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")

if API_KEY:
    print("API Key Attained")
else:
    print("API Key Missing")

API Key Attained


## Testing if the API Key Enables Data Access

In [5]:
CITY = "london"

url = f"https://api.waqi.info/feed/{CITY}/?token={API_KEY}"

response = requests.get(url)

data = response.json()

if data['status'] == 'ok':
    print(f"Data Fetched Successfully for {CITY.capitalize()}!")
    
    current_aqi = data['data']['aqi']
    print(f"Current AQI: {current_aqi}")
    
    print("\nAvailable data sections:")
    for key in data['data'].keys():
        print(f"- {key}")
else:
    print(f" Unable to fetch data. Error: {data.get('data', 'Unknown error')}")

Data Fetched Successfully for London!
Current AQI: 24

Available data sections:
- aqi
- idx
- attributions
- city
- dominentpol
- iaqi
- time
- forecast
- debug


## Pulling Relevant Historical Data

In [7]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_historical_aqi_data(city_name: str, years_back: int = 3) -> pd.DataFrame:
    """
    Resolves coordinates for a target city and fetches historical air quality data.
    """
    # 1. Geocoding
    print(f"Resolving coordinates for target city: {city_name.capitalize()}")
    geocode_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
    
    geo_response = requests.get(geocode_url)
    geo_response.raise_for_status()
    geo_data = geo_response.json()

    if not geo_data.get("results"):
        raise ValueError(f"Coordinates for '{city_name}' could not be resolved.")

    lat = geo_data["results"][0]["latitude"]
    lon = geo_data["results"][0]["longitude"]
    print(f"Coordinates resolved: Latitude {lat}, Longitude {lon}")

    # 2. Define Timeframe
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365 * years_back)
    
    # 3. Fetch Historical Data
    print("Retrieving historical dataset...")
    history_url = (
        f"https://air-quality-api.open-meteo.com/v1/air-quality?"
        f"latitude={lat}&longitude={lon}"
        f"&start_date={start_date.strftime('%Y-%m-%d')}"
        f"&end_date={end_date.strftime('%Y-%m-%d')}"
        f"&hourly=pm10,pm2_5,nitrogen_dioxide,ozone"
    )
    
    hist_response = requests.get(history_url)
    hist_response.raise_for_status()
    hist_data = hist_response.json()
    
    # 4. Parse into Pandas DataFrame
    hourly_data = hist_data.get("hourly", {})
    df = pd.DataFrame({
        "timestamp": pd.to_datetime(hourly_data.get("time", [])),
        "pm10": hourly_data.get("pm10", []),
        "pm2_5": hourly_data.get("pm2_5", []),
        "no2": hourly_data.get("nitrogen_dioxide", []),
        "ozone": hourly_data.get("ozone", [])
    })
    
    # Data Cleaning: Drop rows with missing values
    df = df.dropna().reset_index(drop=True)
    return df

# Execute the extraction
TARGET_CITY = "london"
historical_df = fetch_historical_aqi_data(TARGET_CITY, years_back=3)

print("\nDataset extraction complete.")
print(f"Total historical records extracted: {len(historical_df)}")
print("\nData Preview:")
print(historical_df.head())

Resolving coordinates for target city: London
Coordinates resolved: Latitude 51.50853, Longitude -0.12574
Retrieving historical dataset...

Dataset extraction complete.
Total historical records extracted: 26304

Data Preview:
            timestamp  pm10  pm2_5   no2  ozone
0 2023-04-19 00:00:00  28.2   19.4  11.3   72.0
1 2023-04-19 01:00:00  28.8   19.6   9.9   67.0
2 2023-04-19 02:00:00  27.9   20.6  10.1   63.0
3 2023-04-19 03:00:00  28.3   20.8  11.2   55.0
4 2023-04-19 04:00:00  29.2   22.1  12.1   50.0
